In [ ]:
from SFS.src.py.utils import *
from numpy.fft import fft2, ifft, ifft2, fft, fftfreq, fftshift, ifftshift, rfftfreq, rfft2
from IPython.display import HTML

In [ ]:
number = 9
m = 8
folder = "data/SETD_paper/{n}/{m}/".format(n=number,m=m)
num = count_files(folder)
print(num)

In [ ]:
vid_notebook(folder, 0, skip=1, size=2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from SFS.src.py.utils import get_field, get_para

# Calculate the ensemble average of the mean height, spatial correlation, and variance at final time
def get_weak_metrics(folder_num, M, n_ens):
    # Extract the true dt_0 from the first run's parameters
    _, _, _, _, _, dt0, _ = get_para(f"data/SETD_paper/{folder_num}/1/1/")
    dt_vals = [dt0 / (2**(m-1)) for m in range(1, M+1)]
    
    mean_h_m, corr_m, var_m = [], [], []
    
    for m in range(1, M+1):
        h_seed, corr_seed, var_seed = [], [], []
        for seed in range(1, n_ens+1):
            base_folder = f"data/SETD_paper/{folder_num}/{m}/{seed}/"
            try: field = get_field(base_folder, "varphi")[-1]
            except: print(f"Can't retrieve {seed}"); continue
            h_seed.append(np.mean(field))
            C_dx = np.mean(field * np.roll(field, 1))
            corr_seed.append(C_dx)
            var_seed.append(np.var(field))
            
        mean_h_m.append(np.mean(h_seed))
        corr_m.append(np.mean(corr_seed))
        var_m.append(np.mean(var_seed))
        
    return dt_vals, [np.array(mean_h_m), np.array(corr_m), np.array(var_m)]

# Setup schemes to compare
schemes = [
    {'folder': number+0, 'label': 'ETD1', 'marker': 's'},
    {'folder': number+1, 'label': 'ETD2', 'marker': 'o'},
    {'folder': number+2, 'label': 'IF',   'marker': '^'},
]

# Read M and n_ens automatically
M = len([name for name in os.listdir(f"data/SETD_paper/{number}")])
n_ens = len([name for name in os.listdir(f"data/SETD_paper/{number}/1")])
# M = 6
# n_ens = 32
print(f"Read from data: M={M}, n_ens={n_ens}")

# Collect all data
datas = [[], [], []]
for i, s in enumerate(schemes):
    dt_vals, data = get_weak_metrics(s['folder'], M, n_ens)
    for i, d in enumerate(data):
        datas[i].append(d)

In [ ]:
metrics_to_plot = (
    ('\\varphi(t)', datas[0]),
    ('C(\\mathrm{d}x)', datas[1]),
    ('(\\varphi(t)-\\bar\\varphi(t))^2', datas[2]),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
titles = ["$\\bar \\varphi$", '$C(\\mathrm{d}x)$', '$\\mathrm{Var}(\\varphi)$']

for j, ax  in enumerate(axes):
    (var, data) = metrics_to_plot[j]
    for i, s in enumerate(schemes):
        ax.semilogx(dt_vals, data[i], marker=s['marker'], linestyle='-', label=s['label'])
    
    ax.set_xlabel(r'$\Delta t$')
    ax.set_ylabel(f'$E[{var}]$')
    ax.set_title(f'{titles[j]}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
refs = [d[1][-1] for d in datas]
errs = [[np.abs(dj[:-1] - refs[i]) for dj in d] for i, d in enumerate(datas)]

dt_plot = dt_vals[:-1]
dt_ref = np.array(dt_plot)

for i, err in enumerate(errs):
    plt.figure(figsize=(4, 3))
    for j, s in enumerate(schemes):
        plt.loglog(dt_plot, err[j], marker=s['marker'], linestyle='-', label=s['label'])

    plt.loglog(dt_ref, dt_ref * err[j][0]/dt_ref[0], 'k--', label=r'$\mathcal{O}(\Delta t)$')
    plt.loglog(dt_ref, dt_ref**2 * err[1][0]/dt_ref[0]**2, 'k--', label=r'$\mathcal{O}(\Delta t^2)$') 
    plt.xlabel(r'$\Delta t$')
    plt.title(titles[i])
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()